# ⭐ Day 36: Outlier Detection Methods
## Techniques & Visualizations for AI & ML | Step-by-Step Guide

**Day 36 of 369-day Python & AI Learning Path** 🚀


## Welcome to Day 36!

Welcome to another critical session in our Python & AI Learning Path! Today we're tackling one of the most persistent challenges in real-world machine learning: **outlier detection and handling**. While textbooks often present clean, well-behaved datasets, production data is frequently contaminated with extreme values that can wreak havoc on your models.

Outliers are more than just statistical nuisances—they're signals that demand interpretation. A fraudulent transaction, a sensor malfunction, a data entry error, or a genuine extreme event: each requires different treatment. Mishandling outliers can lead to inflated error metrics, biased parameter estimates, reduced model generalization, and ultimately, failed deployments. In financial modeling, outliers might represent rare but critical market events. In healthcare, they could indicate life-threatening conditions or measurement errors. In manufacturing, they might signal equipment failure.

This notebook equips you with a comprehensive toolkit spanning from classical statistical methods (Z-score, IQR) to modern machine learning approaches (Isolation Forest, LOF, DBSCAN). More importantly, you'll learn *when* to use each method and *how* to decide whether to remove, cap, transform, or retain outliers. By the end of this session, you'll approach outlier detection not as a preprocessing chore, but as a strategic modeling decision that can significantly impact your model's robustness and business value. Let's dive in! 🔍


## 📋 Table of Contents

1. [What Are Outliers and Why They Matter in Machine Learning](#1-what-are-outliers-and-why-they-matter-in-machine-learning)
2. [Types of Outliers](#2-types-of-outliers)
3. [Loading Dataset and Target Variable Overview](#3-loading-dataset-and-target-variable-overview)
4. [Visual Outlier Detection Methods](#4-visual-outlier-detection-methods)
5. [Statistical Methods](#5-statistical-methods)
6. [Advanced Techniques](#6-advanced-techniques)
7. [Comparing Different Detection Methods](#7-comparing-different-detection-methods)
8. [Handling Outliers](#8-handling-outliers-remove-cap-transform-or-keep)
9. [Impact of Outliers on Model Performance](#9-impact-of-outliers-on-model-performance)
10. [Best Practices and Decision Framework](#10-best-practices-and-decision-framework)
11. [🛠️ Hands-On Exercises](#-hands-on-exercises)
12. [Solutions & Key Insights](#-solutions--key-insights)


## 1. What Are Outliers and Why They Matter in Machine Learning ⚠️

### Definition
Outliers are observations that deviate markedly from other observations in a dataset. They lie an abnormal distance from other values in a random sample from a population.

### Why Outliers Are Problematic:

- **📉 Skewed Statistics**: Mean and standard deviation are highly sensitive to outliers
- **📊 Biased Models**: Linear regression, logistic regression, and neural networks are particularly vulnerable
- **📉 Inflated Error Metrics**: MSE and RMSE can become dominated by outlier contributions
- **🎯 Reduced Generalization**: Models trained on contaminated data perform poorly on clean test data
- **⚠️ False Assumptions**: Outliers violate normality assumptions of many statistical tests

### When Outliers Are Valuable:

- **Fraud Detection**: Outliers often represent fraudulent transactions
- **Anomaly Detection**: Equipment failures, network intrusions, rare diseases
- **Risk Modeling**: Extreme market events are precisely what risk models must capture
- **Scientific Discovery**: Novel phenomena often first appear as outliers

**💡 Key Principle**: Never automatically delete outliers without understanding their origin and business context!


## 2. Types of Outliers 📊

Understanding outlier types helps select appropriate detection methods:

### 🔹 By Dimensionality
- **Univariate Outliers**: Extreme values in a single feature (e.g., age = 200 years)
- **Multivariate Outliers**: Normal individually but extreme in combination (e.g., age=25 with income=$1M is unusual even if both values are normal separately)

### 🔹 By Scope
- **Global Outliers**: Points far from the entire dataset (easiest to detect)
- **Contextual/Conditional Outliers**: Normal globally but abnormal in specific context (e.g., 30°C is normal in summer, outlier in winter)
- **Collective Outliers**: A group of points that together form an anomaly (even if individually normal)

### 🔹 By Origin
- **Point Errors**: Data entry mistakes, sensor malfunctions
- **Measurement Errors**: Instrument precision issues
- **Natural Variability**: Genuine extreme events
- **Processing Errors**: ETL pipeline issues, unit conversion errors

**💡 Detection Strategy**: Different types require different methods—univariate methods miss multivariate outliers, global methods miss local anomalies.


In [ ]:
# Import essential libraries for outlier detection
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Set visualization parameters
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")
np.random.seed(42)

print("✅ Libraries loaded successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Scikit-learn available for advanced methods")


In [ ]:
# Create a realistic synthetic dataset with intentional outliers
# Simulating a real estate / house prices dataset

n_samples = 1000

# Generate base features with realistic distributions
data = {
    'square_feet': np.random.lognormal(7.5, 0.5, n_samples),  # ~1800 sq ft average
    'bedrooms': np.random.poisson(3, n_samples).clip(1, 6),
    'bathrooms': np.random.poisson(2, n_samples).clip(1, 4),
    'age_years': np.random.exponential(20, n_samples).clip(0, 100),
    'lot_size': np.random.lognormal(8.5, 0.8, n_samples),
    'distance_to_city': np.random.exponential(15, n_samples).clip(1, 50),
    'school_rating': np.random.beta(7, 2, n_samples) * 10,  # 0-10 scale
    'crime_rate': np.random.exponential(0.05, n_samples).clip(0, 0.5),
    'num_amenities': np.random.poisson(5, n_samples),
    'last_renovation': np.random.choice([0, 5, 10, 15, 20, 25], n_samples)
}

df = pd.DataFrame(data)

# Create target variable (house price) with realistic relationships
base_price = (
    df['square_feet'] * 150 +
    df['bedrooms'] * 25000 +
    df['bathrooms'] * 35000 -
    df['age_years'] * 2000 -
    df['distance_to_city'] * 3000 +
    df['school_rating'] * 15000 -
    df['crime_rate'] * 500000 +
    df['lot_size'] * 20 +
    np.random.normal(0, 50000, n_samples)  # noise
).clip(50000, 2000000)

df['price'] = base_price

# Inject UNIVARIATE OUTLIERS (extreme values in single features)
# 1. Square feet outliers (data entry errors: added extra zero)
outlier_indices_sqft = np.random.choice(df.index, size=15, replace=False)
df.loc[outlier_indices_sqft, 'square_feet'] *= 10

# 2. Age outliers (impossible negative ages or extremely old)
outlier_indices_age = np.random.choice(df.index, size=10, replace=False)
df.loc[outlier_indices_age, 'age_years'] = np.random.choice([150, 200, -5, -10], size=10)

# 3. Price outliers (luxury mansions or data errors)
outlier_indices_price = np.random.choice(df.index, size=12, replace=False)
df.loc[outlier_indices_price[:6], 'price'] *= 5  # Luxury
df.loc[outlier_indices_price[6:], 'price'] = np.random.uniform(5000, 15000, 6)  # Errors

# Inject MULTIVARIATE OUTLIERS (normal individually, extreme together)
# Tiny house with luxury pricing (fraud or data error)
multivariate_outliers = np.random.choice(df.index, size=8, replace=False)
df.loc[multivariate_outliers, 'square_feet'] = np.random.uniform(400, 600, 8)  # Small
df.loc[multivariate_outliers, 'price'] = np.random.uniform(800000, 1200000, 8)  # Expensive

# Contextual outliers: Very high crime rate but extremely high price
contextual_outliers = np.random.choice(df.index, size=6, replace=False)
df.loc[contextual_outliers, 'crime_rate'] = np.random.uniform(0.3, 0.5, 6)
df.loc[contextual_outliers, 'price'] = np.random.uniform(900000, 1500000, 6)

print("✅ Synthetic real estate dataset created with intentional outliers!")
print(f"Dataset shape: {df.shape}")
print(f"Features: {list(df.columns)}")
print(f"\nDataset Overview:")
df.describe()


## 3. Loading Dataset and Target Variable Overview 📋

Let's examine our dataset structure and identify potential issues before formal outlier detection.


In [ ]:
# Comprehensive dataset overview
print("=" * 70)
print("📊 DATASET OVERVIEW")
print("=" * 70)
print(f"Total samples: {len(df):,}")
print(f"Features: {len(df.columns) - 1} (excluding target)")
print(f"Target variable: price (house price)")
print(f"\nFeature Data Types:")
print(df.dtypes)

# Check for obvious issues
print(f"\n⚠️ Potential Issues Detected:")
print(f"Negative ages: {(df['age_years'] < 0).sum()}")
print(f"Ages > 100: {(df['age_years'] > 100).sum()}")
print(f"Extreme square footage (>10000): {(df['square_feet'] > 10000).sum()}")
print(f"Extreme prices (<20000 or >2M): {((df['price'] < 20000) | (df['price'] > 2000000)).sum()}")

# Summary statistics for key features
print(f"\n📈 Target Variable (Price) Statistics:")
print(df['price'].describe())


## 4. Visual Outlier Detection Methods 📉

Visual inspection is always the first step—your eyes can catch patterns algorithms miss.


In [ ]:
# Create comprehensive visual outlier detection plots
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. Box plots for key features
features_to_check = ['square_feet', 'age_years', 'price', 'crime_rate', 'distance_to_city', 'school_rating']

for idx, feature in enumerate(features_to_check):
    ax = axes[idx // 3, idx % 3]
    
    # Box plot
    box_plot = ax.boxplot(df[feature].dropna(), vert=True, patch_artist=True,
                          boxprops=dict(facecolor='lightblue', alpha=0.7),
                          medianprops=dict(color='red', linewidth=2),
                          whiskerprops=dict(color='black', linewidth=1.5),
                          capprops=dict(color='black', linewidth=1.5),
                          flierprops=dict(marker='o', markerfacecolor='red', markersize=5, alpha=0.6))
    
    ax.set_title(f'📦 {feature.replace("_", " ").title()}', fontsize=11, fontweight='bold')
    ax.set_ylabel('Value', fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    
    # Add statistics text
    q1 = df[feature].quantile(0.25)
    q3 = df[feature].quantile(0.75)
    iqr = q3 - q1
    outliers_count = ((df[feature] < (q1 - 1.5 * iqr)) | (df[feature] > (q3 + 1.5 * iqr))).sum()
    ax.text(0.95, 0.95, f'IQR Outliers: {outliers_count}', transform=ax.transAxes, 
            fontsize=9, verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('📦 Box Plots for Outlier Detection\n(Red dots beyond whiskers are IQR outliers)', 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("💡 Insight: Box plots clearly show extreme values in square_feet, age_years, and price!")


In [ ]:
# Violin plots for distribution shape analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

violin_features = ['square_feet', 'price', 'age_years', 'crime_rate']

for idx, feature in enumerate(violin_features):
    ax = axes[idx // 2, idx % 2]
    
    # Violin plot shows distribution shape
    parts = ax.violinplot(df[feature].dropna(), vert=True, showmeans=True, showmedians=True)
    
    # Color the violin
    for pc in parts['bodies']:
        pc.set_facecolor('lightcoral')
        pc.set_alpha(0.7)
    
    parts['cmeans'].set_color('blue')
    parts['cmedians'].set_color('green')
    parts['cmeans'].set_linewidth(2)
    parts['cmedians'].set_linewidth(2)
    
    ax.set_title(f'🎻 {feature.replace("_", " ").title()} Distribution', fontsize=11, fontweight='bold')
    ax.set_ylabel('Value', fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    
    # Add mean and median annotations
    mean_val = df[feature].mean()
    median_val = df[feature].median()
    ax.text(0.95, 0.95, f'Mean: {mean_val:.1f}\nMedian: {median_val:.1f}', 
            transform=ax.transAxes, fontsize=9, verticalalignment='top', 
            horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.suptitle('🎻 Violin Plots: Distribution Shape & Outlier Impact\n(Note skewness caused by outliers)', 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("💡 Insight: Violin plots reveal heavy tails and skewness caused by outliers—especially in price and square_feet!")


In [ ]:
# Scatter plots for multivariate outlier detection
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Price vs Square Feet (should be correlated)
ax1 = axes[0, 0]
ax1.scatter(df['square_feet'], df['price'], alpha=0.6, c='steelblue', edgecolors='black', linewidth=0.5)
ax1.set_xlabel('Square Feet', fontsize=11)
ax1.set_ylabel('Price ($)', fontsize=11)
ax1.set_title('🏠 Price vs Square Feet\n(Multivariate outliers: small area, high price)', fontsize=12, fontweight='bold')
ax1.grid(alpha=0.3)

# Highlight suspicious points (small sqft, high price)
suspicious = df[(df['square_feet'] < 1000) & (df['price'] > 600000)]
if len(suspicious) > 0:
    ax1.scatter(suspicious['square_feet'], suspicious['price'], c='red', s=100, 
                marker='X', label=f'Suspicious ({len(suspicious)})', zorder=5)
    ax1.legend()

# 2. Price vs Age
ax2 = axes[0, 1]
ax2.scatter(df['age_years'], df['price'], alpha=0.6, c='green', edgecolors='black', linewidth=0.5)
ax2.set_xlabel('Age (Years)', fontsize=11)
ax2.set_ylabel('Price ($)', fontsize=11)
ax2.set_title('🏚️ Price vs Age\n(Negative age = data error)', fontsize=12, fontweight='bold')
ax2.grid(alpha=0.3)

# Highlight negative ages
negative_age = df[df['age_years'] < 0]
if len(negative_age) > 0:
    ax2.scatter(negative_age['age_years'], negative_age['price'], c='red', s=100, 
                marker='X', label=f'Negative Age ({len(negative_age)})', zorder=5)
    ax2.legend()

# 3. Crime Rate vs Price (negative correlation expected)
ax3 = axes[1, 0]
ax3.scatter(df['crime_rate'], df['price'], alpha=0.6, c='purple', edgecolors='black', linewidth=0.5)
ax3.set_xlabel('Crime Rate', fontsize=11)
ax3.set_ylabel('Price ($)', fontsize=11)
ax3.set_title('🚔 Crime Rate vs Price\n(High crime + high price = contextual outlier)', fontsize=12, fontweight='bold')
ax3.grid(alpha=0.3)

# Highlight high crime, high price, high_crime_high_price = df[(df['crime_rate'] > 0.3) & (df['price'] > 800000)], if len(high_crime_high_price) > 0:, ax3.scatter(high_crime_high_price['crime_rate'], high_crime_high_price['price'], c='red', s=100, marker='X', label=f'Suspicious ({len(high_crime_high_price)})', zorder=5), ax3.legend(), 
# 4. Distance vs Price (negative correlation expected), ax4 = axes[1, 1], ax4.scatter(df['distance_to_city'], df['price'], alpha=0.6, c='orange', edgecolors='black', linewidth=0.5), ax4.set_xlabel('Distance to City (miles)', fontsize=11), ax4.set_ylabel('Price ($)', fontsize=11), ax4.set_title('📍 Distance vs Price\n(Remote but expensive = suspicious)', fontsize=12, fontweight='bold'), ax4.grid(alpha=0.3), 
plt.suptitle('📊 Scatter Plots for Multivariate Outlier Detection\n(Red X marks suspicious observations)', fontsize=13, fontweight='bold', y=1.02), plt.tight_layout(), 
plt.show(), 
print("💡 Insight: Scatter plots reveal multivariate outliers—instances normal in single features but anomalous in combination!")

## 5. Statistical Methods 📐

Now let's implement formal statistical detection methods with precise thresholds.


In [ ]:
# 1. Z-Score Method (Standard Deviation Method)
def detect_outliers_zscore(df, column, threshold=3):
    """
    Detect outliers using Z-score method
    Z = (X - μ) / σ
    Values with |Z| > threshold are outliers
    """
    z_scores = np.abs(stats.zscore(df[column].dropna()))
    outlier_indices = df[column].dropna().index[z_scores > threshold]
    return outlier_indices, z_scores

# 2. IQR Method (Interquartile Range)
def detect_outliers_iqr(df, column, factor=1.5):
    """
    Detect outliers using IQR method
    Q1 - 1.5*IQR < Normal < Q3 + 1.5*IQR
    """
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - factor * IQR
    upper_bound = Q3 + factor * IQR
    
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers.index, (lower_bound, upper_bound)

# 3. Modified Z-Score (using MAD - Median Absolute Deviation)
def detect_outliers_modified_zscore(df, column, threshold=3.5):
    """
    Modified Z-score using MAD (more robust to outliers)
    MZ = 0.6745 * (X - median) / MAD
    """
    median = df[column].median()
    mad = np.median(np.abs(df[column] - median))
    
    if mad == 0:
        mad = 1e-10  # Avoid division by zero
    
    modified_z_scores = 0.6745 * (df[column] - median) / mad
    outlier_indices = df[np.abs(modified_z_scores) > threshold].index
    
    return outlier_indices, modified_z_scores

# Apply methods to key features
features_to_analyze = ['square_feet', 'age_years', 'price']

print("=" * 70)
print("📊 STATISTICAL OUTLIER DETECTION RESULTS")
print("=" * 70)

results_summary = []

for feature in features_to_analyze:
    print(f"\n🔍 Feature: {feature.upper()}")
    print("-" * 50)
    
    # Z-Score (threshold=3)
    z_outliers, z_scores = detect_outliers_zscore(df, feature, threshold=3)
    print(f"Z-Score Method (|z| > 3):     {len(z_outliers)} outliers ({len(z_outliers)/len(df)*100:.1f}%)")
    
    # IQR (factor=1.5)
    iqr_outliers, bounds = detect_outliers_iqr(df, feature, factor=1.5)
    print(f"IQR Method (1.5×IQR):         {len(iqr_outliers)} outliers ({len(iqr_outliers)/len(df)*100:.1f}%)")
    print(f"  Bounds: [{bounds[0]:.1f}, {bounds[1]:.1f}]")
    
    # Modified Z-Score (threshold=3.5)
    mz_outliers, mz_scores = detect_outliers_modified_zscore(df, feature, threshold=3.5)
    print(f"Modified Z-Score (|mz| > 3.5): {len(mz_outliers)} outliers ({len(mz_outliers)/len(df)*100:.1f}%)")
    
    results_summary.append({
        'Feature': feature,
        'Z-Score_Outliers': len(z_outliers),
        'IQR_Outliers': len(iqr_outliers),
        'Modified_Z_Outliers': len(mz_outliers)
    })

results_df = pd.DataFrame(results_summary)
print(f"\n📋 Summary Table:")
print(results_df.to_string(index=False))


In [ ]:
# Visualize statistical method comparisons
fig, axes = plt.subplots(3, 3, figsize=(15, 12))

for idx, feature in enumerate(features_to_analyze):
    data = df[feature].dropna()
    
    # 1. Original Distribution with Z-Score thresholds
    ax1 = axes[idx, 0]
    ax1.hist(data, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
    mean = data.mean()
    std = data.std()
    ax1.axvline(mean, color='green', linestyle='--', linewidth=2, label=f'Mean: {mean:.1f}')
    ax1.axvline(mean + 3*std, color='red', linestyle='--', linewidth=2, label='±3σ')
    ax1.axvline(mean - 3*std, color='red', linestyle='--', linewidth=2)
    ax1.set_title(f'{feature}\nZ-Score Method', fontsize=10, fontweight='bold')
    ax1.legend(fontsize=8)
    ax1.grid(axis='y', alpha=0.3)
    
    # 2. IQR Method visualization
    ax2 = axes[idx, 1]
    ax2.hist(data, bins=50, alpha=0.7, color='lightcoral', edgecolor='black')
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    ax2.axvline(Q1, color='blue', linestyle='--', linewidth=2, label=f'Q1: {Q1:.1f}')
    ax2.axvline(Q3, color='blue', linestyle='--', linewidth=2, label=f'Q3: {Q3:.1f}')
    ax2.axvline(Q1 - 1.5*IQR, color='red', linestyle='--', linewidth=2, label='Fences')
    ax2.axvline(Q3 + 1.5*IQR, color='red', linestyle='--', linewidth=2)
    ax2.set_title(f'{feature}\nIQR Method', fontsize=10, fontweight='bold')
    ax2.legend(fontsize=8)
    ax2.grid(axis='y', alpha=0.3)
    
    # 3. Modified Z-Score (MAD-based)
    ax3 = axes[idx, 2]
    ax3.hist(data, bins=50, alpha=0.7, color='lightgreen', edgecolor='black')
    median = data.median()
    mad = np.median(np.abs(data - median))
    ax3.axvline(median, color='purple', linestyle='--', linewidth=2, label=f'Median: {median:.1f}')
    ax3.axvline(median + 3.5*mad/0.6745, color='red', linestyle='--', linewidth=2, label='±3.5 MAD')
    ax3.axvline(median - 3.5*mad/0.6745, color='red', linestyle='--', linewidth=2)
    ax3.set_title(f'{feature}\nModified Z-Score', fontsize=10, fontweight='bold')
    ax3.legend(fontsize=8)
    ax3.grid(axis='y', alpha=0.3)

plt.suptitle('📐 Statistical Outlier Detection Methods Comparison\n(Red lines indicate outlier thresholds)', 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("💡 Insight: IQR method is most robust for skewed distributions; Modified Z-score handles extreme outliers better than standard Z-score!")


## 6. Advanced Techniques 🤖

Machine learning-based methods excel at detecting multivariate outliers and handling high-dimensional data.


In [ ]:
# Prepare data for ML-based methods
features_for_ml = ['square_feet', 'bedrooms', 'bathrooms', 'age_years', 
                     'lot_size', 'distance_to_city', 'school_rating', 
                     'crime_rate', 'num_amenities']

X = df[features_for_ml].copy()

# Handle any remaining missing values
X = X.fillna(X.median())

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("✅ Data prepared for ML-based outlier detection")
print(f"Features used: {features_for_ml}")
print(f"Samples: {X_scaled.shape[0]}")
print(f"Dimensions: {X_scaled.shape[1]}")


In [ ]:
# 1. Isolation Forest
print("\n" + "=" * 70)
print("🌲 ISOLATION FOREST")
print("=" * 70)

# Isolation Forest isolates anomalies instead of profiling normal data
iso_forest = IsolationForest(contamination=0.05, random_state=42, n_estimators=100)
iso_labels = iso_forest.fit_predict(X_scaled)
iso_scores = iso_forest.decision_function(X_scaled)

# -1 indicates outlier, 1 indicates inlier
iso_outliers = np.where(iso_labels == -1)[0]

print(f"Contamination parameter: 5%")
print(f"Outliers detected: {len(iso_outliers)} ({len(iso_outliers)/len(df)*100:.1f}%)")
print(f"Average anomaly score: {iso_scores[iso_outliers].mean():.4f}")
print(f"Anomaly score range: [{iso_scores.min():.4f}, {iso_scores.max():.4f}]")

# Add results to dataframe
df['isolation_forest_outlier'] = (iso_labels == -1).astype(int)
df['isolation_forest_score'] = iso_scores


In [ ]:
# 2. Local Outlier Factor (LOF)
print("\n" + "=" * 70)
print("🔍 LOCAL OUTLIER FACTOR (LOF)")
print("=" * 70)

# LOF detects local anomalies based on density deviation
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
lof_labels = lof.fit_predict(X_scaled)
lof_scores = lof.negative_outlier_factor_  # Lower values = more outlying

lof_outliers = np.where(lof_labels == -1)[0]

print(f"Neighbors: 20")
print(f"Outliers detected: {len(lof_outliers)} ({len(lof_outliers)/len(df)*100:.1f}%)")
print(f"LOF score range: [{lof_scores.min():.4f}, {lof_scores.max():.4f}]")
print(f"Mean LOF for outliers: {lof_scores[lof_outliers].mean():.4f}")

df['lof_outlier'] = (lof_labels == -1).astype(int)
df['lof_score'] = lof_scores


In [ ]:
# 3. DBSCAN (Density-Based Spatial Clustering)
print("\n" + "=" * 70)
print("🎯 DBSCAN (Density-Based Clustering)")
print("=" * 70)

# DBSCAN identifies outliers as points that don't belong to any cluster
dbscan = DBSCAN(eps=1.5, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_scaled)

# -1 indicates noise/outlier
dbscan_outliers = np.where(dbscan_labels == -1)[0]
n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)

print(f"EPS parameter: 1.5")
print(f"Min samples: 5")
print(f"Clusters found: {n_clusters}")
print(f"Outliers detected: {len(dbscan_outliers)} ({len(dbscan_outliers)/len(df)*100:.1f}%)")
df['dbscan_outlier'] = (dbscan_labels == -1).astype(int)
print(f"Outlier points added to DataFrame")
# 📊 Comparison of ML Methods:
print(f"Isolation Forest: {len(iso_outliers)} outliers")
print(f"LOF: {len(lof_outliers)} outliers")
print(f"DBSCAN: {len(dbscan_outliers)} outliers")
# Agreement between methods
iso_set = set(iso_outliers)
lof_set = set(lof_outliers)
dbscan_set = set(dbscan_outliers)
common_iso_lof = len(iso_set & lof_set)
common_all = len(iso_set & lof_set & dbscan_set)
print(f"Agreement between Isolation Forest and LOF: {common_iso_lof} points")
# 🤝 Agreement Analysis:
if len(iso_set) > 0:
    iso_lof_pct = common_iso_lof / len(iso_set) * 100
else:
    iso_lof_pct = 0.0

print(f"ISO & LOF agreement: {common_iso_lof} points ({iso_lof_pct:.1f}% of ISO)")
print(f"All three agree:     {common_all} points")

In [ ]:
# Visualize ML-based detection results
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Isolation Forest scores distribution
ax1 = axes[0, 0]
ax1.hist(df['isolation_forest_score'], bins=50, alpha=0.7, color='forestgreen', edgecolor='black')
ax1.axvline(df[df['isolation_forest_outlier'] == 1]['isolation_forest_score'].max(), 
            color='red', linestyle='--', linewidth=2, label='Outlier Threshold')
ax1.set_xlabel('Anomaly Score', fontsize=10)
ax1.set_ylabel('Frequency', fontsize=10)
ax1.set_title('🌲 Isolation Forest\nScore Distribution', fontsize=11, fontweight='bold')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# 2. LOF scores
ax2 = axes[0, 1]
ax2.hist(df['lof_score'], bins=50, alpha=0.7, color='coral', edgecolor='black')
ax2.axvline(df[df['lof_outlier'] == 1]['lof_score'].max(), 
            color='red', linestyle='--', linewidth=2, label='Outlier Threshold')
ax2.set_xlabel('LOF Score (negative)', fontsize=10)
ax2.set_ylabel('Frequency', fontsize=10)
ax2.set_title('🔍 Local Outlier Factor\nScore Distribution', fontsize=11, fontweight='bold')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

# 3. 2D projection with outliers highlighted (PCA)
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

ax3 = axes[1, 0]
normal_mask = df['isolation_forest_outlier'] == 0
outlier_mask = df['isolation_forest_outlier'] == 1

ax3.scatter(X_pca[normal_mask, 0], X_pca[normal_mask, 1], 
            c='lightblue', alpha=0.6, s=30, label='Normal', edgecolors='black', linewidth=0.5)
ax3.scatter(X_pca[outlier_mask, 0], X_pca[outlier_mask, 1], 
            c='red', alpha=0.8, s=100, marker='X', label='ISO Outliers', edgecolors='black')
ax3.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=10)
ax3.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=10)
ax3.set_title('📉 PCA Projection\n(Isolation Forest Outliers)', fontsize=11, fontweight='bold')
ax3.legend()
ax3.grid(alpha=0.3)

# 4. Method agreement Venn diagram (simulated with bar chart)
ax4 = axes[1, 1]
methods = ['Isolation\nForest', 'LOF', 'DBSCAN']
outlier_counts = [len(iso_outliers), len(lof_outliers), len(dbscan_outliers)]
colors = ['forestgreen', 'coral', 'steelblue']

bars = ax4.bar(methods, outlier_counts, color=colors, alpha=0.7, edgecolor='black')
ax4.set_ylabel('Outliers Detected', fontsize=10)
ax4.set_title('📊 Method Comparison\n(Outlier Counts)', fontsize=11, fontweight='bold')
ax4.grid(axis='y', alpha=0.3)

# Add value labels
for bar in bars:
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height + 1,
             f'{int(height)}', ha='center', va='bottom', fontweight='bold')

plt.suptitle('🤖 ML-Based Outlier Detection Results', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("💡 Insight: Isolation Forest and LOF often agree on global outliers but differ on local anomalies!")


## 7. Comparing Different Detection Methods ⚖️

Let's systematically compare all methods to understand their strengths and weaknesses.


In [ ]:
# Comprehensive method comparison
print("=" * 80)
print("⚖️ COMPREHENSIVE METHOD COMPARISON")
print("=" * 80)

# Create comparison dataframe
comparison_data = {
    'Method': ['Z-Score (3σ)', 'IQR (1.5×)', 'Modified Z-Score', 'Isolation Forest', 'LOF', 'DBSCAN'],
    'Type': ['Statistical', 'Statistical', 'Statistical', 'ML/Distance', 'ML/Density', 'Clustering'],
    'square_feet_Outliers': [
        len(detect_outliers_zscore(df, 'square_feet')[0]),
        len(detect_outliers_iqr(df, 'square_feet')[0]),
        len(detect_outliers_modified_zscore(df, 'square_feet')[0]),
        'N/A (Multivariate)',
        'N/A (Multivariate)',
        'N/A (Multivariate)'
    ],
    'price_Outliers': [
        len(detect_outliers_zscore(df, 'price')[0]),
        len(detect_outliers_iqr(df, 'price')[0]),
        len(detect_outliers_modified_zscore(df, 'price')[0]),
        'N/A (Multivariate)',
        'N/A (Multivariate)',
        'N/A (Multivariate)'
    ],
    'Multivariate': ['No', 'No', 'No', 'Yes', 'Yes', 'Yes'],
    'Robust_to_Skew': ['No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes'],
    'Handles_High_Dim': ['Yes', 'Yes', 'Yes', 'Yes', 'Limited', 'Limited'],
    'Computational_Cost': ['Low', 'Low', 'Low', 'Medium', 'High', 'Medium']
}

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

print("\n" + "=" * 80)
print("🎯 METHOD SELECTION GUIDE")
print("=" * 80)
print("""
📊 Use Z-Score when:
   • Data is normally distributed
   • Quick univariate screening needed
   • Outliers are extreme global values

📦 Use IQR when:
   • Data is skewed/non-normal
   • Need robust univariate method
   • Working with financial/economic data

🧮 Use Modified Z-Score when:
   • Extreme outliers present (affects mean/std)
   • Median-based approach preferred
   • Small sample sizes

🌲 Use Isolation Forest when:
   • High-dimensional data
   • Need multivariate detection
   • Large datasets (scalable)

🔍 Use LOF when:
   • Local anomalies matter (context-dependent)
   • Clusters have different densities
   • Moderate dataset size

🎯 Use DBSCAN when:
   • Want clustering + outlier detection
   • Arbitrary cluster shapes expected
   • Density-based definition appropriate
""")


## 8. Handling Outliers: Remove, Cap, Transform, or Keep? 🛠️

Detection is only half the battle—strategic handling is crucial.


In [ ]:
# Demonstrate different outlier treatment strategies
def winsorize_series(series, limits=(0.05, 0.05)):
    """Cap extreme values at percentiles"""
    lower_limit = series.quantile(limits[0])
    upper_limit = series.quantile(1 - limits[1])
    return series.clip(lower=lower_limit, upper=upper_limit)

def log_transform(series):
    """Log transform to reduce skewness"""
    # Add constant to handle zeros/negatives
    offset = abs(series.min()) + 1 if series.min() <= 0 else 0
    return np.log(series + offset)

# Apply different strategies to 'price'
price_original = df['price'].copy()

# Strategy 1: Remove outliers (IQR method)
iqr_outliers, bounds = detect_outliers_iqr(df, 'price', factor=1.5)
price_removed = df.loc[~df.index.isin(iqr_outliers), 'price']

# Strategy 2: Cap outliers (Winsorization at 5% and 95%)
price_capped = winsorize_series(df['price'], limits=(0.05, 0.05))

# Strategy 3: Log transformation
price_log = log_transform(df['price'])

# Create comparison visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Original
ax1 = axes[0, 0]
ax1.hist(price_original, bins=50, alpha=0.7, color='lightblue', edgecolor='black')
ax1.set_title(f'Original Price\nSkewness: {price_original.skew():.2f}', fontsize=11, fontweight='bold')
ax1.set_xlabel('Price ($)')
ax1.set_ylabel('Frequency')
ax1.grid(axis='y', alpha=0.3)

# Remove
ax2 = axes[0, 1]
ax2.hist(price_removed, bins=50, alpha=0.7, color='lightcoral', edgecolor='black')
ax2.set_title(f'Remove Outliers ({len(iqr_outliers)} removed)\nSkewness: {price_removed.skew():.2f}', fontsize=11, fontweight='bold')
ax2.set_xlabel('Price ($)')
ax2.set_ylabel('Frequency')
ax2.grid(axis='y', alpha=0.3)

# Cap
ax3 = axes[1, 0]
ax3.hist(price_capped, bins=50, alpha=0.7, color='lightgreen', edgecolor='black')
ax3.set_title(f'Cap at 5th/95th Percentile\nSkewness: {price_capped.skew():.2f}', fontsize=11, fontweight='bold')
ax3.set_xlabel('Price ($)')
ax3.set_ylabel('Frequency')
ax3.grid(axis='y', alpha=0.3)

# Log transform
ax4 = axes[1, 1]
ax4.hist(price_log, bins=50, alpha=0.7, color='gold', edgecolor='black')
ax4.set_title(f'Log Transformed\nSkewness: {price_log.skew():.2f}', fontsize=11, fontweight='bold')
ax4.set_xlabel('Log(Price)')
ax4.set_ylabel('Frequency')
ax4.grid(axis='y', alpha=0.3)

plt.suptitle('📊 Outlier Treatment Strategies Comparison\n(Lower skewness = more normal distribution)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('treatment_strategies.png', dpi=150, bbox_inches='tight')
plt.show()

print('💡 Insight: Log transformation often outperforms removal/capping by preserving data while normalizing distribution!')

### 📋 Treatment Strategy Decision Matrix

| Strategy | When to Use | Pros | Cons | Example |
|----------|-------------|------|------|---------|
| **🗑️ Remove** | Clear data errors, <5% of data | Clean dataset, improves metrics | Information loss, bias if MNAR | Negative ages, impossible values |
| **🎚️ Cap (Winsorize)** | Extreme but valid values, financial data | Retains observations, reduces impact | Arbitrary thresholds, distorts relationships | Income at 99th percentile |
| **🔄 Transform** | Skewed distributions, positive values | Normalizes, preserves all data | Interpretability loss, fails with zeros/negatives | Log-transform prices |
| **✅ Keep** | Natural variability, anomalies are target | No information loss | May degrade model performance | Fraud detection, rare events |
| **🏷️ Separate Model** | Systematic outliers with different patterns | Captures all signal complexity | Increased complexity, overfitting risk | Luxury vs regular homes |

**💡 Golden Rule**: Always investigate *why* outliers exist before deciding treatment. Domain knowledge trumps statistical rules!


## 9. Impact of Outliers on Model Performance 📉

Let's demonstrate how outliers affect a simple linear regression model.


In [ ]:
# Demonstrate outlier impact on model performance
print("=" * 70)
print("📊 OUTLIER IMPACT ON LINEAR REGRESSION")
print("=" * 70)

# Prepare features
features = ['square_feet', 'bedrooms', 'bathrooms', 'age_years']
X = df[features].copy()
y = df['price'].copy()

# Handle any issues
X = X.fillna(X.median())
X = X.clip(lower=0)  # Remove negative values

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model 1: With outliers
model_with_outliers = LinearRegression()
model_with_outliers.fit(X_train, y_train)
pred_with = model_with_outliers.predict(X_test)
rmse_with = np.sqrt(mean_squared_error(y_test, pred_with))
r2_with = r2_score(y_test, pred_with)

# Identify and remove outliers from training data (IQR method on target)
Q1 = y_train.quantile(0.25)
Q3 = y_train.quantile(0.75)
IQR = Q3 - Q1
outlier_mask = (y_train < (Q1 - 1.5 * IQR)) | (y_train > (Q3 + 1.5 * IQR))

X_train_clean = X_train[~outlier_mask]
y_train_clean = y_train[~outlier_mask]

# Model 2: Without outliers
model_without_outliers = LinearRegression()
model_without_outliers.fit(X_train_clean, y_train_clean)
pred_without = model_without_outliers.predict(X_test)
rmse_without = np.sqrt(mean_squared_error(y_test, pred_without))
r2_without = r2_score(y_test, pred_without)

print(f"Training samples with outliers: {len(X_train)}")
print(f"Training samples without outliers: {len(X_train_clean)} ({len(X_train) - len(X_train_clean)} removed)")
print(f"\n📈 Model Performance Comparison:")
print(f"{'Metric':<20} {'With Outliers':<15} {'Without Outliers':<15} {'Improvement':<15}")
print("-" * 65)
print(f"{'RMSE':<20} {rmse_with:>14,.0f} {rmse_without:>14,.0f} {((rmse_with-rmse_without)/rmse_with*100):>13.1f}%")
print(f"{'R² Score':<20} {r2_with:>14.3f} {r2_without:>14.3f} {((r2_without-r2_with)/r2_with*100):>13.1f}%")

# Visualize predictions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# With outliers
ax1 = axes[0]
ax1.scatter(y_test, pred_with, alpha=0.6, c='blue', edgecolors='black', linewidth=0.5)
ax1.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
ax1.set_xlabel('Actual Price', fontsize=11)
ax1.set_ylabel('Predicted Price', fontsize=11)
ax1.set_title(f'With Outliers\nRMSE: {rmse_with:,.0f}, R²: {r2_with:.3f}', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# Without outliers
ax2 = axes[1]
ax2.scatter(y_test, pred_without, alpha=0.6, c='green', edgecolors='black', linewidth=0.5)
ax2.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
ax2.set_xlabel('Actual Price', fontsize=11)
ax2.set_ylabel('Predicted Price', fontsize=11)
ax2.set_title(f'Without Outliers\nRMSE: {rmse_without:,.0f}, R²: {r2_without:.3f}', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle('📉 Impact of Outlier Removal on Model Predictions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n💡 Insight: Outlier removal improved R² by {((r2_without-r2_with)/r2_with*100):.1f}% - but be careful not to remove legitimate extreme values!")


## 10. Best Practices and Decision Framework 🎯

### 🔑 Golden Rules for Outlier Management:

1. **🔍 Always Visualize First**: Never trust statistics alone—plot your data
2. **🕵️ Investigate Origin**: Determine if outliers are errors, natural variation, or signal
3. **📊 Document Everything**: Record which outliers were removed/modified and why
4. **🧪 Validate Impact**: Compare model performance with/without outlier treatment
5. **🔄 Iterate**: Outlier handling is iterative—refine as you understand the data better

### 🎯 Decision Framework:

```
Step 1: DETECT
   ├─ Visual inspection (box plots, scatter plots)
   ├─ Statistical methods (Z-score, IQR)
   └─ ML methods (Isolation Forest, LOF) for multivariate
   
Step 2: INVESTIGATE
   ├─ Are they data entry errors? → REMOVE
   ├─ Are they measurement errors? → REMOVE/IMPUTE
   ├─ Are they natural extremes? → KEEP or CAP
   └─ Are they the signal you need? → KEEP (fraud/anomaly)
   
Step 3: TREAT
   ├─ Remove (if errors and <5% of data)
   ├─ Cap/Winsorize (if valid extremes)
   ├─ Transform (if skewed distribution)
   └─ Keep with separate model (if systematic differences)
   
Step 4: VALIDATE
   ├─ Compare model metrics before/after
   ├─ Check residual plots for remaining outliers
   └─ Validate on holdout set with natural outliers
```

### ⚠️ Common Pitfalls to Avoid:

- **❌ Blind deletion**: Removing outliers without investigation
- **❌ Inconsistent treatment**: Different rules for train vs test sets
- **❌ Ignoring multivariate outliers**: Focusing only on single features
- **❌ Over-cleaning**: Removing all extreme values destroys variance
- **❌ Data leakage**: Using target variable to detect outliers before splitting

**✅ Remember**: The goal is robust models, not clean data. Sometimes outliers are the most important observations!


## 🛠️ Hands-On Exercises

Apply your outlier detection skills with these progressive exercises!

### Exercise 1: Basic IQR Detection
Implement the IQR method for the `lot_size` feature. Calculate the exact number of outliers and their percentage.


In [ ]:
# Exercise 1: Your code here



### Exercise 2: Z-Score Comparison
Compare Z-score outlier detection with thresholds of 2, 3, and 4 for the `crime_rate` feature. How does the outlier count change? Which threshold seems most appropriate?


In [ ]:
# Exercise 2: Your code here



### Exercise 3: Custom Visualization
Create a side-by-side box plot and violin plot for the `distance_to_city` feature. Add annotations showing the mean, median, and outlier count.


In [ ]:
# Exercise 3: Your code here



### Exercise 4: Multivariate Scatter Analysis
Create a scatter plot of `school_rating` vs `price`. Identify and highlight points where high crime rate (>0.3) coincides with high price (>$800k). These are contextual outliers.


In [ ]:
# Exercise 4: Your code here



### Exercise 5: Isolation Forest Tuning
Run Isolation Forest with contamination values of 0.01, 0.05, and 0.10. Compare how many outliers are detected at each level and visualize the results.


In [ ]:
# Exercise 5: Your code here



### Exercise 6: Winsorization Implementation
Apply 1% and 10% Winsorization to the `square_feet` feature. Compare the distributions before and after using histograms. Calculate how many extreme values were capped at each level.


In [ ]:
# Exercise 6: Your code here



### Exercise 7: Method Agreement Analysis
Find the intersection of outliers detected by IQR (on price), Z-score (on price, threshold=3), and Isolation Forest. Which points do all three methods agree are outliers? Visualize these consensus outliers.


In [ ]:
# Exercise 7: Your code here



### Exercise 8: Outlier Impact on Correlation
Calculate the correlation matrix between `square_feet`, `bedrooms`, and `price` using (a) all data and (b) data with price outliers removed. How do the correlations change? What does this tell you about outlier influence?


In [ ]:
# Exercise 8: Your code here



### Exercise 9: Reusable Detection Function
Write a function `detect_outliers_comprehensive(df, feature, methods=['iqr', 'zscore'])` that returns a DataFrame with outlier flags from each requested method, plus a 'consensus' column indicating how many methods flagged each point.


In [ ]:
# Exercise 9: Your code here



### Exercise 10: Strategic Recommendation
Based on all analysis in this notebook, write a 3-paragraph recommendation for handling outliers in this real estate dataset for a production pricing model. Justify your choices for each feature type and explain your validation approach.


In [ ]:
# Exercise 10: Write your recommendation as comments or print statements



## Solutions & Key Insights (Review After Attempting) 🔑

<details>
<summary>Click to expand solutions after attempting the exercises</summary>

### Exercise 1 Solution
```python
Q1 = df['lot_size'].quantile(0.25)
Q3 = df['lot_size'].quantile(0.75)
IQR = Q3 - Q1
outliers = df[(df['lot_size'] < Q1 - 1.5*IQR) | (df['lot_size'] > Q3 + 1.5*IQR)]
print(f"Outliers: {len(outliers)} ({len(outliers)/len(df)*100:.1f}%)")
```
**Insight**: Lot sizes often have heavy right tail—consider log transformation instead of removal.

### Exercise 2 Solution
```python
for threshold in [2, 3, 4]:
    z_scores = np.abs(stats.zscore(df['crime_rate']))
    outliers = np.sum(z_scores > threshold)
    print(f"Threshold {threshold}: {outliers} outliers")
# Threshold 2 catches too many (normal extremes), 4 catches too few
# Threshold 3 is the standard balance
```
**Insight**: Lower thresholds (2) are too aggressive for non-normal distributions; 3 is standard.

### Exercise 3 Solution
```python
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.boxplot(df['distance_to_city'])
ax1.set_title('Box Plot')
parts = ax2.violinplot(df['distance_to_city'], showmeans=True, showmedians=True)
ax2.set_title(f'Violin Plot\nMean: {df["distance_to_city"].mean():.1f}, Median: {df["distance_to_city"].median():.1f}')
```
**Insight**: Combining both plots shows both summary statistics and distribution shape.

### Exercise 4 Solution
```python
plt.scatter(df['school_rating'], df['price'], alpha=0.6)
suspicious = df[(df['crime_rate'] > 0.3) & (df['price'] > 800000)]
plt.scatter(suspicious['school_rating'], suspicious['price'], c='red', s=100, marker='X')
```
**Insight**: These contextual outliers might indicate data errors or luxury properties in transitioning neighborhoods.

### Exercise 5 Solution
```python
for cont in [0.01, 0.05, 0.10]:
    iso = IsolationForest(contamination=cont, random_state=42)
    labels = iso.fit_predict(X_scaled)
    print(f"Contamination {cont}: {np.sum(labels == -1)} outliers")
# 0.05 is typically a good starting point; adjust based on domain knowledge
```
**Insight**: Contamination is a hyperparameter—tune it based on expected anomaly rate in your domain.

### Exercise 6 Solution
```python
for limit in [0.01, 0.10]:
    capped = df['square_feet'].clip(lower=df['square_feet'].quantile(limit), 
                                     upper=df['square_feet'].quantile(1-limit))
    changed = (capped != df['square_feet']).sum()
    print(f"{limit*100}% Winsorization: {changed} values capped")
# 1% is conservative, 10% is aggressive—choose based on business rules
```
**Insight**: 10% Winsorization may be too aggressive; 1-5% is typical for most applications.

### Exercise 7 Solution
```python
iqr_out, _ = detect_outliers_iqr(df, 'price')
z_out, _ = detect_outliers_zscore(df, 'price')
iso_out = np.where(iso_forest.fit_predict(X_scaled) == -1)[0]
consensus = set(iqr_out) & set(z_out) & set(iso_out)
print(f"Consensus outliers: {len(consensus)}")
# These are the strongest outliers—safe to investigate first
```
**Insight**: Consensus outliers across methods are the strongest candidates for investigation/removal.

### Exercise 8 Solution
```python
corr_all = df[['square_feet', 'bedrooms', 'price']].corr()
df_clean = df[~df.index.isin(detect_outliers_iqr(df, 'price')[0])]
corr_clean = df_clean[['square_feet', 'bedrooms', 'price']].corr()
print("Correlation change:", corr_clean - corr_all)
# Outliers inflate correlations—removing them gives more realistic estimates
```
**Insight**: Outliers often inflate correlations; "clean" correlations better represent typical relationships.

### Exercise 9 Solution
Your function should iterate through methods, apply detection, store binary flags, and sum across methods for consensus. Return a DataFrame with original data + method columns + consensus count.

### Exercise 10 Solution
**Paragraph 1**: For `age_years`, remove impossible values (negative, >200) as data errors. For `square_feet`, apply 2% Winsorization to handle data entry errors (extra zeros) while preserving legitimate large homes.

**Paragraph 2**: For `price`, use Isolation Forest for multivariate detection rather than univariate methods, as luxury homes may have extreme prices but valid feature combinations. Create a separate flag for high-price outliers to allow the model to learn luxury segment patterns.

**Paragraph 3**: Validate by comparing RMSE on a holdout set with and without outlier treatment. Monitor for data leakage—never use target variable statistics from test set for outlier detection. Document all decisions in model cards for auditability.

</details>
